# Feature Importance & Interpretability

Understand which features drive the at-risk prediction using built-in feature importance and SHAP values.

> **Prerequisite**: Run notebook 05 to save the best model to `models/best_model.joblib`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split

from src.data import DATA_DIR
from src.preprocessing import build_preprocessor
from src.utils import load_model

%matplotlib inline

## 1. Load Model & Data

In [ ]:
model = load_model("best_model.joblib")

df = pd.read_csv(DATA_DIR / "student-mat-engineered.csv")
X = df.drop(columns=["at_risk"])
y = df["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Get transformed feature names
preprocessor = model.named_steps["pre"]
X_test_transformed = preprocessor.transform(X_test)

# Build feature names from the preprocessor
num_names = preprocessor.transformers_[0][2]  # numeric column names
cat_encoder = preprocessor.named_transformers_["cat"]
cat_names = cat_encoder.get_feature_names_out().tolist()
feature_names = list(num_names) + cat_names

print(f"Model type: {type(model.named_steps['clf']).__name__}")
print(f"Features: {len(feature_names)}")

## 2. Built-in Feature Importance

Available for tree-based models (Random Forest, XGBoost, LightGBM).

In [ ]:
clf = model.named_steps["clf"]

if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=True)

    # Plot top 15
    fig, ax = plt.subplots(figsize=(8, 6))
    fi.tail(15).plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
    ax.set_xlabel("Feature Importance")
    ax.set_title("Top 15 Features (Built-in Importance)")
    plt.tight_layout()
    plt.show()
else:
    print("Model does not have built-in feature_importances_. Skipping.")

## 3. SHAP Analysis

In [ ]:
# Create a SHAP explainer on the classifier with transformed data
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_transformed)

# For binary classifiers, shap_values may be a list of two arrays;
# use the positive class (index 1) if so.
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

### 3a. SHAP Summary Plot (Beeswarm)

In [ ]:
shap.summary_plot(shap_vals, X_test_transformed, feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.show()

### 3b. SHAP Bar Plot (Global Importance)

In [ ]:
shap.summary_plot(shap_vals, X_test_transformed, feature_names=feature_names, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (mean |SHAP|)")
plt.tight_layout()
plt.show()

### 3c. SHAP Force Plots (Individual Predictions)

One at-risk student and one not-at-risk student.

In [ ]:
shap.initjs()

y_pred = model.predict(X_test)

# Find one at-risk and one not-at-risk prediction
at_risk_idx = np.where(y_pred == 1)[0][0]
not_risk_idx = np.where(y_pred == 0)[0][0]

print("At-risk student:")
display(shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    shap_vals[at_risk_idx],
    feature_names=feature_names,
))

print("\nNot-at-risk student:")
display(shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    shap_vals[not_risk_idx],
    feature_names=feature_names,
))